Import thư viện & Cấu hình

In [4]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K

# --- CẤU HÌNH ---
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 20        # Custom CNN nhẹ hơn nên có thể tăng Epochs lên 20-30
NUM_FOLDS = 5      # Số vòng lặp K-Fold
LEARNING_RATE = 0.001 # Tốc độ học (Custom CNN thường dùng 0.001, VGG hay dùng 0.0001)

# --- TỰ ĐỘNG PHÁT HIỆN MÔI TRƯỜNG ---
try:
    from google.colab import drive
    print("Detected: GOOGLE COLAB environment")
    drive.mount('/content/drive')
    # SỬA ĐƯỜNG DẪN NÀY THEO FOLDER TRÊN DRIVE CỦA BẠN
    DATA_DIR = '/content/drive/MyDrive/AI_VGG16_Classifier/data/raw' 
except ImportError:
    print("Detected: LOCAL environment")
    DATA_DIR = '../data/processed' 

print(f"Đang tìm dữ liệu tại: {DATA_DIR}")

Detected: LOCAL environment
Đang tìm dữ liệu tại: ../data/processed


Tải danh sách ảnh vào DataFrame

In [5]:
def load_image_paths(data_dir):
    image_dir = Path(data_dir)
    # Tìm tất cả các đuôi ảnh phổ biến
    filepaths = list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png')) + list(image_dir.glob(r'**/*.jpeg'))
    
    # Lấy tên thư mục cha làm nhãn (Label)
    labels = [os.path.split(os.path.split(filepath)[0])[1] for filepath in filepaths]

    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')

    df = pd.concat([filepaths, labels], axis=1)
    
    # Trộn ngẫu nhiên (Shuffle)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# Chạy hàm tải dữ liệu
try:
    df = load_image_paths(DATA_DIR)
    print(f"Tổng số ảnh tìm thấy: {len(df)}")
    print("\nSố lượng ảnh mỗi lớp:")
    print(df['Label'].value_counts())
except Exception as e:
    print(f"Lỗi tìm ảnh: {e}")

Tổng số ảnh tìm thấy: 274

Số lượng ảnh mỗi lớp:
Label
pins_Ronaldo     101
pins_Messi        92
pins_Benzenma     81
Name: count, dtype: int64


Xây dựng kiến trúc Custom CNN

In [6]:
def build_custom_cnn(num_classes):
    model = Sequential()
    
    # --- BLOCK 1 ---
    # Input nhận ảnh 224x224x3
    model.add(Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3)))
    # Conv2D: Trích xuất đặc trưng
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    # MaxPooling: Giảm kích thước ảnh đi một nửa
    model.add(MaxPooling2D((2, 2)))
    
    # --- BLOCK 2 ---
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    
    # --- BLOCK 3 ---
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    
    # --- PHẦN PHÂN LOẠI (CLASSIFIER) ---
    model.add(Flatten()) # Duỗi phẳng dữ liệu
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5)) # Tắt ngẫu nhiên 50% neuron để chống học vẹt (Overfitting)
    
    # Lớp đầu ra (Output Layer)
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

print("Đã khởi tạo hàm build_custom_cnn thành công!")

Đã khởi tạo hàm build_custom_cnn thành công!


Huấn luyện K-Fold

In [7]:
# Khởi tạo K-Fold
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

# --- CẤU HÌNH DATA GENERATOR (Quan trọng: Rescale 1./255) ---
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Chuẩn hóa pixel về [0, 1]
    rotation_range=20,      # Xoay ảnh ngẫu nhiên
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255          # Chỉ chuẩn hóa, không xoay lật
)

acc_per_fold = []
loss_per_fold = []
fold_no = 1

# BẮT ĐẦU VÒNG LẶP
for train_index, val_index in kf.split(df):
    print(f"\nTraining Custom CNN for Fold {fold_no} ...")
    
    train_data = df.iloc[train_index]
    val_data = df.iloc[val_index]
    
    # 1. Tạo Train Generator
    train_gen = train_datagen.flow_from_dataframe(
        train_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    
    # Lấy danh sách lớp đầy đủ để tránh lỗi thiếu class
    full_classes = list(train_gen.class_indices.keys())
    
    # 2. Tạo Val Generator (Ép buộc dùng full_classes)
    val_gen = val_datagen.flow_from_dataframe(
        val_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False,
        classes=full_classes 
    )
    
    # 3. Gọi hàm xây dựng Custom CNN
    num_classes = len(full_classes)
    model = build_custom_cnn(num_classes)
    
    # 4. Callbacks (Lưu file tên khác để không đè lên model VGG cũ)
    checkpoint_path = f"../models/custom_cnn_fold_{fold_no}.h5"
    if 'google.colab' in str(get_ipython()):
         checkpoint_path = f"/content/drive/MyDrive/models/custom_cnn_fold_{fold_no}.h5"

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
    
    # 5. Training
    try:
        history = model.fit(
            train_gen,
            epochs=EPOCHS, 
            validation_data=val_gen,
            callbacks=callbacks
        )
        
        # Ghi nhận kết quả
        scores = model.evaluate(val_gen, verbose=0)
        print(f'-> Kết quả Fold {fold_no}: Accuracy = {scores[1]*100:.2f}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])
        
    except Exception as e:
        print(f"Lỗi tại Fold {fold_no}: {e}")

    # Dọn dẹp RAM
    K.clear_session()
    fold_no += 1

# Báo cáo cuối cùng
print("\n" + "="*30)
if len(acc_per_fold) > 0:
    print(f"TRUNG BÌNH CỘNG (Custom CNN): {np.mean(acc_per_fold):.2f}%")
else:
    print("Chưa chạy xong fold nào.")
print("="*30)


Training Custom CNN for Fold 1 ...
Found 219 validated image filenames belonging to 3 classes.
Found 55 validated image filenames belonging to 3 classes.
Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3416 - loss: 1.7544
Epoch 1: val_accuracy improved from None to 0.34545, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.3562 - loss: 1.5676 - val_accuracy: 0.3455 - val_loss: 1.1004
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4044 - loss: 1.0852
Epoch 2: val_accuracy did not improve from 0.34545
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4064 - loss: 1.0815 - val_accuracy: 0.3455 - val_loss: 1.0316
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4871 - loss: 1.0351
Epoch 3: val_accuracy did not improve from 0.34545
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4886 - loss: 1.0119 - val_accuracy: 0.3455 - val_loss: 0.9922
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4524 - loss: 0.9970
Epoch 4: val_accuracy improved from 0.34545 to 0.98182, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4521 - loss: 1.0075 - val_accuracy: 0.9818 - val_loss: 0.9059
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5029 - loss: 0.9731
Epoch 5: val_accuracy did not improve from 0.98182
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.5525 - loss: 0.9293 - val_accuracy: 0.8909 - val_loss: 0.7622
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6571 - loss: 0.8081
Epoch 6: val_accuracy did not improve from 0.98182
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.6301 - loss: 0.7738 - val_accuracy: 0.9455 - val_loss: 0.5336
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7379 - loss: 0.6300
Epoch 7: val_accuracy improved from 0.98182 to 1.00000, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.7352 - loss: 0.6066 - val_accuracy: 1.0000 - val_loss: 0.2938
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8071 - loss: 0.4202
Epoch 8: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8584 - loss: 0.3937 - val_accuracy: 0.9818 - val_loss: 0.2466
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8556 - loss: 0.3964
Epoch 9: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8219 - loss: 0.4500 - val_accuracy: 1.0000 - val_loss: 0.1564
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8745 - loss: 0.3892
Epoch 10: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8676 - loss: 0.3834 - val_accuracy: 1.0000 - val_loss: 0.1487
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9388 - loss: 0.2558
Epoch 11: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━


Training Custom CNN for Fold 2 ...
Found 219 validated image filenames belonging to 3 classes.
Found 55 validated image filenames belonging to 3 classes.
Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3790 - loss: 3.4001
Epoch 1: val_accuracy improved from None to 0.36364, saving model to ../models/custom_cnn_fold_2.h5


...

7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.3318 - loss: 1.6773 - val_accuracy: 0.4630 - val_loss: 1.0831
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4309 - loss: 1.0764
Epoch 2: val_accuracy did not improve from 0.46296
7/7 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.4091 - loss: 1.0771 - val_accuracy: 0.2963 - val_loss: 1.0729
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3840 - loss: 1.1027
Epoch 3: val_accuracy did not improve from 0.46296
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.4000 - loss: 1.1032 - val_accuracy: 0.2963 - val_loss: 1.0491
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4846 - loss: 1.0374
Epoch 4: val_accuracy improved from 0.46296 to 0.81481, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.4864 - loss: 1.0113 - val_accuracy: 0.8148 - val_loss: 0.8632
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4956 - loss: 0.9833
Epoch 5: val_accuracy did not improve from 0.81481
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.5000 - loss: 0.9778 - val_accuracy: 0.6852 - val_loss: 0.7582
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6402 - loss: 0.8226
Epoch 6: val_accuracy improved from 0.81481 to 0.96296, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.6273 - loss: 0.8254 - val_accuracy: 0.9630 - val_loss: 0.5872
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7426 - loss: 0.6929
Epoch 7: val_accuracy did not improve from 0.96296
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.6636 - loss: 0.8172 - val_accuracy: 0.6111 - val_loss: 0.9949
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5704 - loss: 0.9942
Epoch 8: val_accuracy did not improve from 0.96296
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.6136 - loss: 0.8549 - val_accuracy: 0.9259 - val_loss: 0.6205
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6840 - loss: 0.7294
Epoch 9: val_accuracy improved from 0.96296 to 0.98148, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.7273 - loss: 0.6928 - val_accuracy: 0.9815 - val_loss: 0.4421
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6693 - loss: 0.6370
Epoch 10: val_accuracy did not improve from 0.98148
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.6591 - loss: 0.6630 - val_accuracy: 0.9815 - val_loss: 0.4052
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8136 - loss: 0.5394
Epoch 11: val_accuracy improved from 0.98148 to 1.00000, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.8000 - loss: 0.5342 - val_accuracy: 1.0000 - val_loss: 0.2734
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8161 - loss: 0.4439
Epoch 12: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8182 - loss: 0.4314 - val_accuracy: 0.9815 - val_loss: 0.1811
Epoch 13/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8007 - loss: 0.4442
Epoch 13: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.7955 - loss: 0.4546 - val_accuracy: 0.9630 - val_loss: 0.2109
Epoch 14/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7708 - loss: 0.5229
Epoch 14: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8045 - loss: 0.4845 - val_accuracy: 0.9444 - val_loss: 0.2348
Epoch 15/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8938 - loss: 0.3693
Epoch 15: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━